# M2 Ablation — Run Multiple Generators on the Same Eval Set

Notebook equivalent of `python -m src.cli.ablate`. Runs the consistency
scorer with each chosen M2 against the same `consistency_eval*.jsonl`,
saves per-M2 score JSONLs, prints AUC + Cohen's d, and writes the ablation
table to `outputs/figures/ablation_table.csv`.

This is the **headline result table** for the writeup.


In [ ]:
# [COLAB_BOOTSTRAP_v2_modular]
# Each Colab session:
#   1. Mount Drive             (Drive has the repo + the tar.gz of M1 features)
#   2. cd into the repo        (so relative paths to src/, configs/, etc. work)
#   3. Symlink Drive data dirs (processed/, qvh_frames/, etc.) into the repo
#   4. EXTRACT M1 features tar.gz from Drive ->  LOCAL SSD  (/content/features_local)
#      Reason: 19,000 small .npz files corrupt on Drive's FUSE; LOCAL SSD is fast.
#   5. Clone upstream moment_detr code (for build_model imports)
#
# Drive is ONLY used as the SOURCE of the tar.gz. Features ALWAYS live on
# /content/ (LOCAL SSD) after extraction. Same pattern as
# run_pipeline_allinone.ipynb cell 2.5 and sanity_consistency.ipynb cell 0.

import os, sys, subprocess, time, tarfile, shutil
from pathlib import Path

# ----- 1. Mount Drive ----------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# ----- 2. Find repo + cd ------------------------------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive/thumbnail_project')
REPO_DIR   = DRIVE_ROOT / 'thumbnail-consistency'
assert REPO_DIR.exists(), f"Upload your repo folder to {REPO_DIR} first."

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ----- 3. Symlink data dirs ---------------------------------------------
for d in ['processed', 'outputs', 'qvh_videos', 'qvh_frames', 'qvh_thumbnails',
          'qvh_frames_within_neg', 'qvh_frames_multi', 'checkpoints']:
    src, dst = DRIVE_ROOT / d, REPO_DIR / d
    if src.exists() and not dst.exists() and not dst.is_symlink():
        try: os.symlink(src, dst)
        except OSError: pass

# ----- 4. Extract M1 features from Drive .tar.gz -> LOCAL SSD ----------
# Source on Drive (the 7.5 GB archive):
ARCHIVE_ON_DRIVE = DRIVE_ROOT / 'moment_detr_features.tar.gz'
# Destination on LOCAL SSD (NOT on Drive — Drive can't hold 19k small files):
EXTRACTED_LOCAL  = Path('/content/features_local')

def _find_feature_dirs(root):
    """Look inside `root` for clip_features/ and clip_text_features/ dirs
    that contain .npz files. Returns (clip_dir, text_dir) paths or (None, None)."""
    clip = text = None
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            p = Path(r) / d
            if d == 'clip_features' and len(list(p.glob('*.npz'))) > 1000:
                clip = p
            if d == 'clip_text_features' and len(list(p.glob('*.npz'))) > 1000:
                text = p
    return clip, text

# Check ONLY local SSD (NOT Drive) for an already-extracted copy from earlier
# in THIS Colab session. If you restart the runtime, /content is wiped and
# this check returns (None, None), triggering a fresh extraction.
CLIP_FEAT_DIR, TEXT_FEAT_DIR = (None, None)
if EXTRACTED_LOCAL.exists():
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(EXTRACTED_LOCAL)

if CLIP_FEAT_DIR and TEXT_FEAT_DIR:
    print(f"[features] Found local extraction from earlier this session (skip).")
    print(f"           {EXTRACTED_LOCAL}")
else:
    assert ARCHIVE_ON_DRIVE.exists(), (
        f"Source archive missing on Drive: {ARCHIVE_ON_DRIVE}\n"
        f"This is the only file the bootstrap reads from Drive."
    )
    # Wipe any stale partial extraction and start fresh
    if EXTRACTED_LOCAL.exists():
        shutil.rmtree(EXTRACTED_LOCAL)
    EXTRACTED_LOCAL.mkdir(parents=True)
    sz_gb = ARCHIVE_ON_DRIVE.stat().st_size / 1e9
    print(f"[features] Extracting {sz_gb:.2f} GB:")
    print(f"           FROM (Drive)     : {ARCHIVE_ON_DRIVE}")
    print(f"           TO   (local SSD) : {EXTRACTED_LOCAL}")
    t0 = time.time()
    with tarfile.open(ARCHIVE_ON_DRIVE) as tf:
        tf.extractall(EXTRACTED_LOCAL)
    print(f"[features] Done in {time.time()-t0:.0f}s.")
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(EXTRACTED_LOCAL)

assert CLIP_FEAT_DIR and TEXT_FEAT_DIR, (
    f"Feature dirs not found after extraction. Check {EXTRACTED_LOCAL}"
)

# ----- 5. Clone upstream Moment-DETR (for build_model imports) ---------
MD_DIR = Path('/content/moment_detr_upstream')
if not MD_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jayleicn/moment_detr.git',
                    str(MD_DIR)], check=True)
if str(MD_DIR) not in sys.path:
    sys.path.insert(0, str(MD_DIR))

# ----- 6. Verification summary -----------------------------------------
import torch
print()
print(f"Repo            : {REPO_DIR}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu (WARN: slow)'}")
print(f"Drive tar.gz    : {ARCHIVE_ON_DRIVE}  ({sz_gb:.2f} GB)" if 'sz_gb' in dir() else "")
print(f"clip_features   : {CLIP_FEAT_DIR}  ({len(list(CLIP_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"clip_text_feats : {TEXT_FEAT_DIR}  ({len(list(TEXT_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"src/ importable : {(REPO_DIR / 'src' / '__init__.py').exists()}")


## 1. Install M2 dependencies (only if your M2_LIST contains a VLM)

Skip this cell if `M2_LIST` only contains `gt`, `random`, and/or `clipcap`.


In [ ]:
import sys, subprocess
USE_VLM = True   # ★ set False to skip VLM deps
if USE_VLM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers>=4.45', 'accelerate>=0.30', 'bitsandbytes>=0.43',
                    'qwen-vl-utils', 'sentencepiece'], check=True)
    print('VLM deps installed.')
else:
    print('Skipped VLM deps.')


## 2. Configure the ablation

In [ ]:
import yaml

# ★ EDIT THESE TWO LINES to change the experiment
EVAL_PATH    = 'processed/consistency_eval.jsonl'         # also: consistency_eval_hard / _query_mismatch
M2_LIST      = ['gt', 'random']                            # add 'clipcap', 'qwen2vl_simple', etc.
K_QUERIES    = 3

# Per-generator kwargs
M2_KWARGS = {
    'gt'             : {'lookup_jsonl': EVAL_PATH},
    'random'         : {'pool_jsonl'  : EVAL_PATH, 'seed': 2026},
    'clipcap'        : {'ckpt_path'   : 'checkpoints/clipcap_best.pt'},
    'qwen2vl_simple' : {'model_id': 'Qwen/Qwen2-VL-7B-Instruct',          'load_in_4bit': True},
    'qwen2vl_aspect' : {'model_id': 'Qwen/Qwen2-VL-7B-Instruct',          'load_in_4bit': True},
    'llava16_aspect' : {'model_id': 'llava-hf/llava-v1.6-mistral-7b-hf',  'load_in_4bit': True},
}

cfg_yaml = yaml.safe_load(open('configs/consistency.yaml'))
print(f'Eval set      : {EVAL_PATH}')
print(f'M2 generators : {M2_LIST}')
print(f'K queries     : {K_QUERIES}')
print(f'alpha / tau / top_p : {cfg_yaml["alpha"]} / {cfg_yaml["tau"]} / {cfg_yaml["top_p"]}')


## 2. Build M1 + scorer (once, shared across all M2 runs)

In [ ]:
from src.models.moment_detr import MomentDETRFeaturePredictor
from src.scoring.consistency import ConsistencyScorer, ScoringConfig

m1 = MomentDETRFeaturePredictor(
    ckpt_path=cfg_yaml['paths']['m1_ckpt'],
    repo_root=str(REPO_DIR),
)
scoring_cfg = ScoringConfig(
    alpha=cfg_yaml['alpha'], tau=cfg_yaml['tau'], top_p=cfg_yaml['top_p'],
    chunk_window=cfg_yaml['chunk_window'], chunk_stride=cfg_yaml['chunk_stride'],
)
scorer = ConsistencyScorer(m1, str(CLIP_FEAT_DIR), scoring_cfg)
print('M1 + scorer ready.')


## 3. Load the eval pairs

In [ ]:
import json
pairs = [json.loads(l) for l in open(EVAL_PATH, encoding='utf-8') if l.strip()]
print(f'Loaded {len(pairs)} pairs from {EVAL_PATH}')
print(f'  positives: {sum(1 for p in pairs if p["label"] == 1)}')
print(f'  negatives: {sum(1 for p in pairs if p["label"] == 0)}')


## 4. Run the ablation — one M2 at a time

Each M2 takes one of these paths:
- **lightweight** (`gt`, `random`, `clipcap`): seconds to load, < 1s/pair
- **heavy** (Qwen, LLaVA): 1-2 min to load, 3-7 sec/pair

Heavy M2s should be run one at a time in this notebook (or restart the runtime
between them) to avoid OOM. Use the GPU memory cleanup cell below if needed.


In [ ]:
from tqdm.auto import tqdm
from src.m2_generators import build_m2
from src.eval.metrics import summarize_separation
from pathlib import Path
import gc, torch

OUT_DIR = REPO_DIR / 'outputs' / 'predictions'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ablation_rows = []

for m2_name in M2_LIST:
    print(f'\n========== {m2_name} ==========')
    m2 = build_m2(m2_name, **M2_KWARGS.get(m2_name, {}))

    results = []
    for r in tqdm(pairs, desc=m2_name):
        # gt needs qid; others ignore extra kwargs
        gen_kwargs = {'qid': r['thumb_qid']} if m2_name == 'gt' else {}
        try:
            qs = m2.generate(r['frame_path'], k=K_QUERIES, **gen_kwargs)
            out = scorer.score(
                thumbnail_path=r['frame_path'],
                vid=r['video_vid'],
                queries=qs,
            )
            results.append({'pair_id': r['pair_id'], 'label': r['label'],
                            'score': float(out['C']), 'queries': qs})
        except Exception as e:
            print(f'  FAIL pair {r["pair_id"]}: {e}')

    # Save per-M2 predictions
    out_path = OUT_DIR / f'scores_{m2_name}.jsonl'
    with open(out_path, 'w', encoding='utf-8') as f:
        for x in results:
            f.write(json.dumps(x, ensure_ascii=False) + '\n')
    print(f'  saved -> {out_path} ({len(results)} pairs)')

    # Summary
    scores = [x['score'] for x in results]
    labels = [x['label'] for x in results]
    stats  = summarize_separation(scores, labels)
    stats['m2']        = m2_name
    stats['n_scored']  = len(results)
    ablation_rows.append(stats)
    print(f"  AUC = {stats['auc']:.4f}   Cohen's d = {stats['cohens_d']:.3f}")

    # Free GPU before loading the next M2
    del m2
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 5. Build the ablation table

In [ ]:
import pandas as pd

df = pd.DataFrame(ablation_rows)
keys = ['m2', 'n_scored', 'n_pos', 'n_neg',
        'mean_pos', 'mean_neg', 'auc', 'cohens_d', 'welch_t', 'welch_p']
df = df[keys].set_index('m2').round(4)

fig_dir = REPO_DIR / 'outputs' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
csv_path = fig_dir / f'ablation_table_{Path(EVAL_PATH).stem}.csv'
df.to_csv(csv_path)
print(f'Saved {csv_path}')
df


## 6. Histogram per M2 (saved + displayed)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(M2_LIST), figsize=(4.5 * len(M2_LIST), 4), sharey=True)
if len(M2_LIST) == 1:
    axes = [axes]

for ax, m2_name in zip(axes, M2_LIST):
    rows = [json.loads(l) for l in open(OUT_DIR / f'scores_{m2_name}.jsonl', encoding='utf-8') if l.strip()]
    pos = [r['score'] for r in rows if r['label'] == 1]
    neg = [r['score'] for r in rows if r['label'] == 0]
    auc = df.loc[m2_name, 'auc']
    ax.hist(pos, bins=25, alpha=0.55, label=f'pos (n={len(pos)})', color='tab:green')
    ax.hist(neg, bins=25, alpha=0.55, label=f'neg (n={len(neg)})', color='tab:red')
    ax.set_title(f'{m2_name}\nAUC = {auc:.3f}', fontsize=10)
    ax.set_xlabel('C(T, V)')
    ax.legend(fontsize=8)
plt.tight_layout()
out_png = fig_dir / f'ablation_{Path(EVAL_PATH).stem}.png'
plt.savefig(out_png, dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved {out_png}')
